In [16]:
from langchain_ollama import ChatOllama
model= ChatOllama(model="lamma3.2:latest")
print("Model is Ready")

Model is Ready


In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Prompt template
user_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}")
])



In [23]:
def get_response(user_input, history):
    # Convert Gradio history to LangChain history
    chat_history = []

    for message in history:
        if message["role"] == "user":
            chat_history.append(
                HumanMessage(content=message["content"])
            )

        elif message["role"] == "assistant":
            chat_history.append(
                AIMessage(content=message["content"])
            )

    # Create prompt with PAST conversation + CURRENT question
    prompt = user_prompt_template.format_messages(
        chat_history=chat_history,
        user_input=user_input
    )

    # Call Ollama / LangChain model
    response = model.invoke(prompt)

    # Add current user message
    history.append({
        "role": "user",
        "content": user_input
    })

    # Add AI response
    history.append({
        "role": "assistant",
        "content": response.content
    })

    # Return complete history
    return history


In [25]:
import gradio as gr

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.messages import (
    HumanMessage,
    AIMessage
)


# --------------------------------------------------
# 1. Create the prompt template
# --------------------------------------------------

user_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}")
])


# --------------------------------------------------
# 2. Function to generate response
# --------------------------------------------------

def get_response(user_input, history):

    # Make sure history exists
    if history is None:
        history = []

    # ----------------------------------------------
    # Convert Gradio history to LangChain messages
    # ----------------------------------------------

    chat_history = []

    for user_message, assistant_message in history:

        if user_message:
            chat_history.append(
                HumanMessage(content=user_message)
            )

        if assistant_message:
            chat_history.append(
                AIMessage(content=assistant_message)
            )

    # ----------------------------------------------
    # Create the prompt
    # ----------------------------------------------

    user_prompt = user_prompt_template.format_messages(
        chat_history=chat_history,
        user_input=user_input
    )

    # ----------------------------------------------
    # Send prompt to Ollama/LangChain model
    # ----------------------------------------------

    response = model.invoke(user_prompt)

    # ----------------------------------------------
    # Add new conversation to Gradio history
    # ----------------------------------------------

    history.append(
        (user_input, response.content)
    )

    # ----------------------------------------------
    # Return complete history
    # ----------------------------------------------

    return history


# --------------------------------------------------
# 3. Create Gradio interface
# --------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown(
        "## 💬 Chat with Ollama Model"
    )

    chatbot = gr.Chatbot()

    msg = gr.Textbox(
        placeholder="Type your message here...",
        label="Your Message"
    )

    clear = gr.Button("Clear Chat")


    # ----------------------------------------------
    # Submit message
    # ----------------------------------------------

    msg.submit(
        get_response,
        inputs=[msg, chatbot],
        outputs=chatbot
    )


    # ----------------------------------------------
    # Clear conversation
    # ----------------------------------------------

    clear.click(
        lambda: [],
        inputs=None,
        outputs=chatbot,
        queue=False
    )


# --------------------------------------------------
# 4. Launch application
# --------------------------------------------------

if __name__ == "__main__":

    demo.launch(
        server_name="0.0.0.0",
        server_port=7866
    )


* Running on local URL:  http://0.0.0.0:7866
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\kakno\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\kakno\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\route_utils.py", line 410, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "c:\Users\kakno\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\blocks.py", line 2330, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "c:\Users\kakno\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\blocks.py", line 1690, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
      